> **ARCHIVED research notebook** - kept for provenance and understanding only.
> It is **superseded by the `cloaq` package** (`src/cloaq/`) and may contain known bugs; see [`research/legacy-notebooks/README.md`](../README.md).
> Saved outputs were cleared and local paths removed. QASMBench paths are relative to your own checkout.

In [ ]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.library import CXGate
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2
import random
import numpy as np
from math import pi
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_aer import AerSimulator

 
# Get a fake backend from the fake provider
# backend = FakeManilaV2()
backend = AerSimulator()

In [ ]:
def remove_idle_qwires(circuit):
    dag = circuit_to_dag(circuit)
    idle_qubits = list(dag.idle_wires())

    # Identify active qubits
    active_qubits = [q for q in dag.qubits if q not in idle_qubits]

    # Create a new circuit with same number of classical bits
    new_circuit = QuantumCircuit(len(active_qubits), circuit.num_clbits)

    # Map qubits: original → new
    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(active_qubits)}

    # Map clbits (1-to-1 since all classical bits are retained)
    clbit_map = {old: new for old, new in zip(circuit.clbits, new_circuit.clbits)}

    for instr, qargs, cargs in circuit.data:
        # Only include instructions on active qubits
        if all(q in qubit_map for q in qargs):
            new_qargs = [qubit_map[q] for q in qargs]
            new_cargs = [clbit_map[c] for c in cargs]
            new_circuit.append(instr, new_qargs, new_cargs)

    return new_circuit

def remove_qk_register(circuit):
    """
    Returns a copy of the circuit without the qk register,
    dropping any gates that involve qk.
    """
    qk_qubit = circuit.qregs[-1][0]
    
    # Build new circuit with all registers except qk
    new_qregs = [reg for reg in circuit.qregs if reg.name != 'qk']
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {q: new_circuit.qubits[i] 
                 for i, q in enumerate(circuit.qubits) 
                 if q != qk_qubit}
    clbit_map = {old: new_circuit.clbits[i] 
                 for i, old in enumerate(circuit.clbits)}

    for inst in circuit.data:
        # Skip any instruction that involves qk
        if qk_qubit in inst.qubits:
            continue
        if all(q in qubit_map for q in inst.qubits):
            new_qargs = [qubit_map[q] for q in inst.qubits]
            new_cargs = [clbit_map[c] for c in inst.clbits]
            new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
qc = QuantumCircuit.from_qasm_file('./QASMBench/small/adder_n4/adder_n4.qasm')

qc.draw('mpl')


In [ ]:
# Look into the transpile function and see if there are any options to preserve the original circuit structure, 
# See how to expand 
#transpiled_circuit = transpile(qc, backend)
transpiled_circuit = transpile(
    qc,
    backend,
    optimization_level=0,           # 0 = no optimization, preserves structure  
    basis_gates=None,               # don't force decomposition into specific gate set
    coupling_map=None,              # ignore device topology constraints
    layout_method='trivial',        # keep qubits in original order, no remapping
    routing_method='none',          # no qubit routing/swapping
    translation_method='translator' # minimal gate translation
)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_unlocked = result_sim.get_counts(transpiled_circuit)
print(counts_unlocked)
plot_histogram(counts_unlocked)

In [ ]:
# Draw Transpiled Ciruit
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
transpiled_circuit_simp.draw('mpl')  

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import Barrier, RZGate
import random
import math
import numpy as np

def transform_circuit(qc, seed=42, num_logic_keys=6, verbose=True):
    """
    Integrated QLL + Phase-Wall Locking (CLOAQ)
    Fixed: Synchronized key tracking, correct H/X state transitions BEFORE the gate, 
           dummy logic separation, and integrated per-gate phase randomization.
    """
    if seed is not None:
        random.seed(seed)

    phase_gates = {'rz', 'p', 's', 't', 'sdg', 'tdg', 'z', 'u1'}

    # Strip existing barriers
    orig_instrs = [
        inst for inst in qc.data
        if not isinstance(inst.operation, Barrier)
    ]

    # Find valid indices for logic obfuscation (non-phase gates)
    non_phase_indices = [
        i for i, inst in enumerate(orig_instrs)
        if inst.operation.name.lower() not in phase_gates
    ]

    # --- Setup Logic Key & Slots ---
    n = min(num_logic_keys, len(non_phase_indices))
    logic_key = [random.randint(0, 1) for _ in range(n)]
    
    # Select exactly 'n' indices where we will inject the logic locks
    key_slots = sorted(random.sample(non_phase_indices, n)) if n > 0 else []

    # --- Build Locked Circuit ---
    key_qreg = QuantumRegister(1, 'qk')
    locked_qc = QuantumCircuit(*qc.qregs, key_qreg, *qc.cregs)
    
    qubit_map = {q: locked_qc.qubits[i] for i, q in enumerate(qc.qubits)}
    clbit_map = {old: locked_qc.clbits[i] for i, old in enumerate(qc.clbits)}
    qk = key_qreg[0]

    slot_ptr = 0
    expected_qk_state = 0  # qk starts in |0>
    last_phase = None
    phase_key = []

    for idx, inst in enumerate(orig_instrs):
        name = inst.operation.name.lower()
        is_phase = name in phase_gates

        # Phase-wall barrier on layer type transition
        if last_phase is not None and is_phase != last_phase:
            locked_qc.barrier(*locked_qc.qubits)

        # ----------------------------------------------------------------
        # 1. Logic Obfuscation
        # ----------------------------------------------------------------
        if not is_phase and slot_ptr < n and idx == key_slots[slot_ptr]:
            key_bit = logic_key[slot_ptr]
            target = qubit_map[inst.qubits[-1]] # Usually the target is the last qubit

            # Apply H unconditionally to separate key boundaries
            locked_qc.h(qk)

            # Apply X BEFORE the gate if the required state differs from current
            if key_bit != expected_qk_state:
                locked_qc.x(qk)
                expected_qk_state = key_bit

            if key_bit == 1:
                # Real Lock: Promote to controlled gate
                controlled = inst.operation.control(1)
                locked_qc.append(
                    controlled,
                    [qk] + [qubit_map[q] for q in inst.qubits],
                    [clbit_map[c] for c in inst.clbits]
                )
            else:
                # Dummy Lock: Dummy CX, then append original unmodified gate
                locked_qc.cx(qk, target)
                locked_qc.append(
                    inst.operation,
                    [qubit_map[q] for q in inst.qubits],
                    [clbit_map[c] for c in inst.clbits]
                )
            
            slot_ptr += 1
            last_phase = is_phase
            continue

        # ----------------------------------------------------------------
        # 2. Phase Obfuscation (Missing in original script)
        # ----------------------------------------------------------------
        if is_phase:
            # Extract kappa (3-bit key) and randomize angle
            if inst.operation.params:
                original_angle = float(inst.operation.params[0])
                kappa = int(round(original_angle / (math.pi / 4))) % 8
                phase_key.append(kappa)
                
                # Assign a new random angle, excluding the correct one
                candidates = [k for k in range(8) if k != kappa]
                rand_angle = random.choice(candidates) * (math.pi / 4)
                
                if name == 'rz':
                    locked_qc.rz(rand_angle, qubit_map[inst.qubits[0]])
                elif name == 'p':
                    locked_qc.p(rand_angle, qubit_map[inst.qubits[0]])
                else:
                    # Pass through fixed phase gates
                    locked_qc.append(inst.operation, [qubit_map[q] for q in inst.qubits], [clbit_map[c] for c in inst.clbits])
            else:
                locked_qc.append(inst.operation, [qubit_map[q] for q in inst.qubits], [clbit_map[c] for c in inst.clbits])
                
            last_phase = is_phase
            continue

        # ----------------------------------------------------------------
        # 3. Default Pass-Through (Unmodified non-phase gates)
        # ----------------------------------------------------------------
        locked_qc.append(
            inst.operation,
            [qubit_map[q] for q in inst.qubits],
            [clbit_map[c] for c in inst.clbits]
        )
        
        # Optional: Add barriers after H gates to match your original styling
        if name == 'h':
            locked_qc.barrier(*[qubit_map[q] for q in inst.qubits])
            
        last_phase = is_phase

    # Final barrier
    if last_phase:
        locked_qc.barrier(*locked_qc.qubits)

    if verbose:
        print(f"Logic Key: {logic_key}")
        print(f"Phase Key: {phase_key} (kappa values 0-7)")
        print(f"Key slots: {key_slots}")

    return locked_qc, logic_key, phase_key

In [ ]:
converted_circuit, key, meta = transform_circuit(qc)
converted_circuit.draw('mpl')

In [ ]:

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(converted_circuit, backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_original = result_sim.get_counts(transpiled_circuit)
print(counts_original)
plot_histogram(counts_original)

In [ ]:
def key_phase(key):
    """
    Maps a 3-bit binary string to a specific phase angle (in radians) based on π.

    Parameters:
    -----------
    key : str
        A 3-bit binary string (e.g., '000', '011', '111').

    Returns:
    --------
    float
        A phase angle in radians, ranging from 0.25π to 2π, depending on the input key.

    Mapping:
    --------
        '000' → 0.25π
        '001' → 0.5π
        '010' → 0.75π
        '011' → 1.0π
        '100' → 1.25π
        '101' → 1.5π
        '110' → 1.75π
        '111' → 2.0π
    """
    match key:
        case '000':
            return 0.25 * pi
        case '001':
            return 0.5 * pi
        case '010':
            return 0.75 * pi
        case '011':
            return 1.0 * pi
        case '100':
            return 1.25 * pi
        case '101':
            return 1.5 * pi
        case '110':
            return 1.75 * pi
        case '111':
            return 2.0 * pi

In [ ]:
from math import pi
from qiskit import QuantumCircuit
from qiskit.circuit import ClassicalRegister 
from qiskit import QuantumRegister
import random


def replace_with_phase_gates(circuit, key_phase_offset=0):
    """
    Replace Z, S, T, and their inverses with custom phase gates.
    Compatible with QLL-locked circuits (preserves qk register and CX gates).
    """
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    qubit_map = {old: new for old, new in zip(circuit.qubits, new_circuit.qubits)}
    clbit_map = {old: new for old, new in zip(circuit.clbits, new_circuit.clbits)}

    for instruction in circuit.data:               
        gate   = instruction.operation
        qubits = instruction.qubits
        clbits = instruction.clbits

        q_new = [qubit_map[q] for q in qubits]
        c_new = [clbit_map[c] for c in clbits] if clbits else []

        if gate.name == "z":
            new_circuit.p(pi + key_phase_offset, q_new[0])
        elif gate.name == "s":
            new_circuit.p(pi/2 + key_phase_offset, q_new[0])
        elif gate.name == "t":
            new_circuit.p(pi/4 + key_phase_offset, q_new[0])
        elif gate.name == "sdg":
            new_circuit.p(-pi/2 + key_phase_offset, q_new[0])
        elif gate.name == "tdg":
            new_circuit.p(-pi/4 + key_phase_offset, q_new[0])
        else:
            new_circuit.append(gate, q_new, c_new)

    return new_circuit

In [ ]:
phase_converted_circuit = replace_with_phase_gates(converted_circuit)
phase_converted_circuit.draw('mpl')

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

def add_measurements_from_info(circuit, measurement_info, exclude_qk=True):
    """
    Adds measurement gates to a quantum circuit based on the provided measurement information.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to which measurements will be added.
    measurement_info (dict): A dictionary containing measurement information, as returned by `get_measurement_info`.
    exclude_qk (bool): If True, don't measure the last qubit (assumed to be qk).

    Returns:
    QuantumCircuit: The modified quantum circuit with measurement gates added.
    """
    # Identify qk if excluding
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        # Assume qk is the last register 
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])
    
    # Iterate over the measurement information
    for qubit, info in measurement_info.items():
        qubit_index = info['measured_qubit']
        clbit_index = info['classical_bit']
        
        # Skip qk if excluding
        if exclude_qk and qubit_index == qk_index:
            continue
        
        # Add the measurement gate to the circuit
        circuit.measure(qubit_index, clbit_index)
    
    return circuit


def remove_all_measurements(circuit):
    """
    Removes all measurement operations from a quantum circuit,
    preserving all quantum and classical registers exactly.
    """
    new_qregs = [QuantumRegister(reg.size, reg.name) for reg in circuit.qregs]
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {}
    for old_reg, new_reg in zip(circuit.qregs, new_qregs):
        for i in range(old_reg.size):
            qubit_map[old_reg[i]] = new_reg[i]

    clbit_map = {}
    for old_reg, new_reg in zip(circuit.cregs, new_cregs):
        for i in range(old_reg.size):
            clbit_map[old_reg[i]] = new_reg[i]

    for instruction in circuit.data:
        inst = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits
        
        if inst.name == 'measure':
            continue

        new_q = [qubit_map[q] for q in qargs]
        new_c = [clbit_map[c] for c in cargs]

        new_circuit.append(inst, new_q, new_c)

    return new_circuit


def get_measurement_info(circuit, exclude_qk=True):
    """
    Analyzes a quantum circuit and provides information about measurement gates.
    Optionally excludes the key qubit (qk) from the returned info.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to analyze.
    exclude_qk (bool): If True, don't include qk measurements in the returned info.

    Returns:
    dict: A dictionary containing information about the measurement gates.
    """
    measurement_info = {}
    
    # Identify qk if excluding
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])
    
    for instruction in circuit.data:
        inst = instruction.operation
        qubits = instruction.qubits
        clbits = instruction.clbits
        
        if inst.name == 'measure':
            qubit_index = circuit.find_bit(qubits[0]).index
            clbit_index = circuit.find_bit(clbits[0]).index
            
            # Skip qk if excluding
            if exclude_qk and qubit_index == qk_index:
                continue
            
            measurement_info[qubit_index] = {
                'measured_qubit': qubit_index,
                'measurement_gate': 'M',
                'classical_bit': clbit_index
            }
    
    return measurement_info


measurement_info = get_measurement_info(phase_converted_circuit, exclude_qk=True)

print("Measurement Information (excluding qk):")
for qubit, info in measurement_info.items():
    print(f"Qubit {info['measured_qubit']} is measured using gate '{info['measurement_gate']}' "
          f"and the result is stored in classical bit {info['classical_bit']}.")

In [ ]:
def add_random_phase_layers_with_reassembly(circuit, num_layers, num_splits):
    """
    Inserts random phase layers at split points.
    Compatible with QLL-locked circuits (preserves qk register).
    """
    phases = [0.25*pi, 0.5*pi, 0.75*pi, 1*pi, 1.25*pi, 1.5*pi, 1.75*pi, 2*pi]

    new_qregs = [QuantumRegister(qreg.size, qreg.name) for qreg in circuit.qregs]
    new_cregs = [ClassicalRegister(creg.size, creg.name) for creg in circuit.cregs]

    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {}
    for old_reg, new_reg in zip(circuit.qregs, new_qregs):
        for i in range(old_reg.size):
            qubit_map[old_reg[i]] = new_reg[i]

    clbit_map = {}
    for old_reg, new_reg in zip(circuit.cregs, new_cregs):
        for i in range(old_reg.size):
            clbit_map[old_reg[i]] = new_reg[i]

    ops = [
        (inst.operation, inst.qubits, inst.clbits)
        for inst in circuit.data
        if inst.operation.name != "measure"
    ]

    split_points = [len(ops) // (num_splits + 1) * (i + 1) for i in range(num_splits)]

    last_gate_name = None

    def safe_append(inst, q, c):
        nonlocal last_gate_name

        if inst.name == "barrier":
            if last_gate_name != "barrier":
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"
            return

        if inst.name == last_gate_name and inst.name in ("h", "cx"):
            return

        q_new = [qubit_map[x] for x in q]
        c_new = [clbit_map[x] for x in c]

        new_circuit.append(inst, q_new, c_new)
        last_gate_name = inst.name

    for i, (inst, q, c) in enumerate(ops):
        safe_append(inst, q, c)

        if i + 1 in split_points:
            for _ in range(num_layers):
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"
                for qb in new_circuit.qubits:
                    new_circuit.p(random.choice(phases), qb)
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"

    # Identify qk (last register) and measure all qubits except it
    qk_qubit = new_circuit.qregs[-1][0]
    for i, qb in enumerate(new_circuit.qubits):
        if qb != qk_qubit:
            new_circuit.measure(qb, i)

    return new_circuit


random_layer_added_qc = add_random_phase_layers_with_reassembly(remove_all_measurements(phase_converted_circuit), 1, 2)
random_layer_added_qc.draw('mpl')

In [ ]:
random_layer_added_qc = add_random_phase_layers_with_reassembly(remove_all_measurements(phase_converted_circuit), 1,2)
random_layer_added_qc.draw('mpl')

In [ ]:
# Transpile the circuit using the pass manager
transpiled_circuit = transpile(random_layer_added_qc, backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_locked = result_sim.get_counts(transpiled_circuit)
print(counts_locked)
plot_histogram(counts_locked)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import RemoveBarriers, DAGFixedPoint
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit import Barrier  

# PHASE-WALL LOCKING 
def modify_rz_inside_barriers(circuit: QuantumCircuit, phase_shift: float):
    """
    Adds a phase shift to all RZ gates inside barrier regions.
    This is the ACTIVE LOCK used before decryption.
    """

    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(circuit.qubits)}
    clbit_map = {old: new_circuit.clbits[i] for i, old in enumerate(circuit.clbits)}

    inside_barrier = False

    for inst in circuit.data:
        op = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        if op.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.barrier(*[qubit_map[q] for q in qargs])
            continue

        if inside_barrier and op.name == 'rz':
            new_circuit.rz(op.params[0] + phase_shift, qubit_map[qargs[0]])
        else:
            new_circuit.append(op, [qubit_map[q] for q in qargs],
                               [clbit_map[c] for c in cargs])

    return new_circuit


# PHASE-WALL DECRYPTION
def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
    """
    Inverse of modify_rz_inside_barriers.
    Subtracts the phase offset from Phase / RZ gates inside barriers.
    """

    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(circuit.qubits)}
    clbit_map = {old: new_circuit.clbits[i] for i, old in enumerate(circuit.clbits)}
    inside_barrier = False

    for inst in circuit.data:
        instruction = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        # Handle barriers
        if instruction.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.append(instruction, [qubit_map[q] for q in qargs], [clbit_map[c] for c in cargs])
            continue

        # Handle phase gates INSIDE barriers
        if inside_barrier and instruction.name in ['rz', 'p']:
            theta = instruction.params[0]

            if instruction.name == 'rz':
                new_circuit.rz(theta - offset, qargs[0])
            else:  # 'p'
                new_circuit.p(theta - offset, qargs[0])

            continue

        # Default: copy gate
        new_circuit.append(instruction, qargs, cargs)

    return new_circuit


# QLL DECRYPTION 
def decrypt_qll(locked_circuit: QuantumCircuit, qll_key) -> QuantumCircuit:
    """
    QLL Decryption following the paper's Dec(Circ', k) process:

    - Removes H gates on qk
    - Inserts Pauli-X gates on qk wherever key bits differ
      (this prepares qk in the correct classical state before each CX)
    - Keeps all CX gates in place (controlled by qk)
    - Does NOT remove qk — the circuit retains the key qubit and CX structure
    
    After this step the CX gates are deterministic:
      qk == |1> => CX acts as X on target  
      qk == |0> => CX acts as identity  
    """

    qk = locked_circuit.qregs[-1][0]

    # Output circuit keeps ALL registers including the key qubit
    qc = QuantumCircuit(*locked_circuit.qregs, *locked_circuit.cregs)

    qubit_map = {q: qc.qubits[i] for i, q in enumerate(locked_circuit.qubits)}
    clbit_map = {old: qc.clbits[i] for i, old in enumerate(locked_circuit.clbits)}

    key_ptr = 0
    # Track whether we just handled a dummy double-CX
    last_cx_was_dummy = False

    for instruction in locked_circuit.data:
        op = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits

        # Drop barriers
        if op.name == "barrier":
            continue

        # Remove H gates on qk
        if op.name == "h" and qk in qargs:
            continue

        if op.name == "x" and qk in qargs:
            continue

        if op.name == "cx" and qargs[0] == qk:
            if last_cx_was_dummy:
                # This is the second CX of a dummy pair — skip it, don't advance key_ptr
                last_cx_was_dummy = False
                continue

            if key_ptr < len(qll_key):
                key_bit = qll_key[key_ptr]
                if key_bit == 1:
                    qc.x(qubit_map[qargs[1]])
                    last_cx_was_dummy = False
                else:
                    # Dummy: next CX belongs to this pair
                    last_cx_was_dummy = True
                key_ptr += 1
            continue
        
        if qk in qargs:
            continue

        new_qargs = [qubit_map[q] for q in qargs]
        new_cargs = [clbit_map[c] for c in cargs]
        qc.append(op, new_qargs, new_cargs)

    return qc

# FULL DECRYPTION PIPELINE (QLL + PHASE-WALL)
def decrypt_full_circuit(
    locked_circuit: QuantumCircuit,
    qll_key,
    phase_offset: float
):
    """
    Fully decrypts a circuit locked with:
      - QLL control-path locking
      - Barrier-based phase-wall locking
    """

    # Step 1: QLL unlock
    qc = decrypt_qll(locked_circuit, qll_key)

    # Step 2: Phase-wall unlock
    qc = subtract_phase_offset(qc, phase_offset)

    # Step 3: Barrier cleanup
    pm = PassManager(RemoveBarriers())
    qc = pm.run(qc)
    
    # Step 4: Strip qk register from final output
    qk = qc.qregs[-1][0]
    new_qregs = [reg for reg in qc.qregs if reg.name != 'qk']
    new_circuit = QuantumCircuit(*new_qregs, *qc.cregs)

    qubit_map = {q: new_circuit.qubits[i]
                 for i, q in enumerate(qc.qubits)
                 if q != qk}
    clbit_map = {old: new_circuit.clbits[i]
                 for i, old in enumerate(qc.clbits)}

    for inst in qc.data:
        if all(q == qk for q in inst.qubits):
            continue
        new_qargs = [qubit_map[q] for q in inst.qubits]
        new_cargs = [clbit_map[c] for c in inst.clbits]
        new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
circuit_for_locking = remove_qk_register(transpiled_circuit_simp)

modified_circuit, qll_key, lock_info = transform_circuit(circuit_for_locking, seed=42)
print(f"qregs after transform: {modified_circuit.qregs}")

p_offset = key_phase("010")
modified_circuit = modify_rz_inside_barriers(modified_circuit, -p_offset)
print(f"qregs after modify_rz: {modified_circuit.qregs}")

decrypted_circuit = decrypt_full_circuit(
    locked_circuit = modified_circuit,
    qll_key = qll_key,
    phase_offset = p_offset   
)

decrypted_circuit.draw('mpl')


In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import random
import time

# TVD Function Definition 
def total_variation_distance_from_counts(counts1, counts2, shots=1024, plot=True, labels=None):
    """
    Calculate TVD per the paper's formula:

        TVD = sum_i( |yi,lock - yi,orig| ) / (2 * N * 2^b)

    Where:
        N   = number of shots
        b   = number of bits in the output (inferred from state length)
        yi  = raw count of outcome i in each distribution
    
    Parameters:
    -----------
    counts1 : dict
        First distribution (e.g., original circuit counts)
    counts2 : dict
        Second distribution (e.g., locked circuit counts)
    shots : int
        Number of shots used in the simulation
    plot : bool
        If True, display histograms of both distributions
    labels : list of str
        Labels for the two distributions (default: ["Distribution 1", "Distribution 2"])
    
    Returns:
    --------
    float : TVD value
    """
    
    if labels is None:
        labels = ["Distribution 1", "Distribution 2"]

    # All possible output states across both distributions
    all_states = set(counts1.keys()).union(set(counts2.keys()))

    # Infer number of bits from state string length
    b = len(next(iter(all_states)))

    # Map states to indices
    state_to_index = {state: i for i, state in enumerate(all_states)}

    # Fill raw count vectors (missing states = 0 counts)
    p = np.zeros(len(all_states))
    q = np.zeros(len(all_states))

    for state, count in counts1.items():
        p[state_to_index[state]] = count
    for state, count in counts2.items():
        q[state_to_index[state]] = count

    # TVD = sum|yi,lock - yi,orig| / (2 * N * 2^b)
    tvd = np.sum(np.abs(p - q)) / (2 * shots * (2 ** b))

    if plot:
        fig = plot_histogram([counts1, counts2], legend=labels, figsize=(12, 6))
        display(fig)  
         

    return tvd 

backend = AerSimulator()


In [ ]:
# Step 1: Simulate ORIGINAL circuit
transpiled_original = transpile(circuit_for_locking, backend)
job_original = backend.run(transpiled_original, shots=1024)
counts_original = job_original.result().get_counts(transpiled_original)

# Step 2: Fix measurements on already-decrypted circuit (reuse from previous cell)
decrypted_clean = remove_all_measurements(decrypted_circuit)
measurement_info = get_measurement_info(circuit_for_locking, exclude_qk=False)
decrypted_clean = add_measurements_from_info(decrypted_clean, measurement_info, exclude_qk=False)

# Step 3: Simulate DECRYPTED circuit
transpiled_decrypted = transpile(decrypted_clean, backend)
job_decrypted= backend.run(transpiled_decrypted, shots=1024)
counts_decrypted = job_decrypted.result().get_counts(transpiled_decrypted)

# Step 4: Calculate TVD
tvd = total_variation_distance_from_counts(
    counts_original,
    counts_decrypted,
    shots=1024,
    plot=True,
    labels=["Original Circuit", "Decrypted Circuit"]
)

print("Counts Original: ", counts_original)
print("Counts Decrypted:", counts_decrypted)
print(f"Total Variation Distance: {tvd:.6f}")
print(f"Key: {qll_key}")